Muhammad Adam Hasani - 25200013 - Tugas 1 Data Wrangling

Bagian A

1. Pemetaan kebutuhan bisnis ke kebutuhan teknis

Data yang dicari ada di tiga tempat:
1. CSV dari kasir: nomor invoice, tanggal, jumlah barang, harga, kode kupon
2. Database CRM: data pelanggan, tipe membership, poin loyalty, status aktif
3. API logistik: status pengiriman, kurir

2. Matriks RACI

Peran: HOBS = Head of Business Strategy, DMO = Data Management Office, TW = Tim Data Wrangling, BI = Business Intelligence Analyst
R = yang kerja, A = yang tanggung jawab akhir, C = ditanya, I = diberi tahu

Menentukan kebutuhan informasi: HOBS = A, DMO = C, TW = I, BI = C
Menetapkan standar data: HOBS = I, DMO = A, TW = C, BI = I
Ekstraksi dan integrasi data: HOBS = I, DMO = C, TW = R, BI = I
Validasi kualitas data: HOBS = I, DMO = A, TW = R, BI = I
Pembuatan kamus data: HOBS = I, DMO = A, TW = R, BI = C
Analisis performa kupon: HOBS = A, DMO = I, TW = C, BI = R

Bagian B.1

In [1]:
import pandas as pd
import numpy as np

df_pos = pd.read_csv(
    'transaksi_pos_nrm.csv',
    parse_dates=['InvoiceDate'],
    dtype={'InvoiceNo': str, 'StockCode': str}
)

In [2]:
df_pos.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total_amount,coupon_code,coupon_discount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,NaN,NaN
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,NaN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,NaN,NaN
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,NaN
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,NaN


In [3]:
df_pos.info()
print('invoice:', df_pos['InvoiceNo'].nunique())
print('customer:', df_pos['CustomerID'].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   InvoiceNo        80000 non-null  str           
 1   StockCode        80000 non-null  str           
 2   Description      80000 non-null  str           
 3   Quantity         80000 non-null  int64         
 4   InvoiceDate      80000 non-null  datetime64[us]
 5   UnitPrice        80000 non-null  float64       
 6   CustomerID       80000 non-null  int64         
 7   Country          80000 non-null  str           
 8   total_amount     80000 non-null  float64       
 9   coupon_code      4171 non-null   str           
 10  coupon_discount  4171 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(5)
memory usage: 6.7 MB
invoice: 4005
customer: 1904


Bagian B.2

In [4]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///customers_nrm.db')

df_crm = pd.read_sql("""SELECT customer_id, email, gender, age, city,
                           membership_tier, loyalty_points, is_active
                    FROM customers
                    WHERE is_active = 1""", engine)

df_crm.head()

,customer_id,email,gender,age,city,membership_tier,loyalty_points,is_active
0,12346,customer12346@mail.com,L,19,Yogyakarta,Gold,2006,1
1,12347,customer12347@mail.com,L,61,Yogyakarta,Bronze,4837,1
2,12348,customer12348@mail.com,L,23,Bandung,Silver,4139,1
3,12350,customer12350@mail.com,L,63,Yogyakarta,Platinum,1805,1
4,12352,customer12352@mail.com,P,18,Semarang,Silver,3462,1


In [5]:
print('pelanggan aktif:', len(df_crm), 'dari 1904')
df_crm['membership_tier'].value_counts()

pelanggan aktif: 1755 dari 1904


membership_tier
Platinum    458
Silver      450
Gold        434
Bronze      413
Name: count, dtype: int64

Bagian B.3

In [6]:
import requests, json

url = 'https://raw.githubusercontent.com/adamhasani/tugas1-data-wrangling/master/status_pengiriman.json'

try:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    api_data = resp.json()
except Exception:
    with open('status_pengiriman.json') as f:
        api_data = json.load(f)

In [7]:
print(json.dumps(api_data['data'][0], indent=2))

{
  "order_id": "536365",
  "shipment": {
    "tracking_number": "NRM37C9CE5E5E",
    "courier": "AnterAja",
    "service": "YES"
  },
  "status": "Delivered",
  "timeline": {
    "shipped_at": "2011-02-13 00:00:00",
    "delivered_at": "2011-02-16 00:00:00"
  },
  "customer_id": 17850
}


In [8]:
df_logistik = pd.json_normalize(api_data['data'], sep='_')
print(df_logistik.shape)
df_logistik.head()

(4005, 8)


,order_id,status,customer_id,shipment_tracking_number,shipment_courier,shipment_service,timeline_shipped_at,timeline_delivered_at
0,536365,Delivered,17850,NRM37C9CE5E5E,AnterAja,YES,2011-02-13 00:00:00,2011-02-16 00:00:00
1,536366,Delivered,17850,NRMF197DB58BD,SiCepat,SameDay,2011-02-28 00:00:00,2011-03-05 00:00:00
2,536367,Shipped,13047,NRM7BA21B6ED2,SiCepat,Economy,2011-01-27 00:00:00,NaN
3,536368,Delivered,13047,NRM5F5642120F,Ninja,Economy,2011-01-13 00:00:00,2011-01-20 00:00:00
4,536369,Delivered,13047,NRMFCBB6C31D2,AnterAja,Economy,2011-02-09 00:00:00,2011-02-12 00:00:00


Bagian B.4

1. dedup logistik dan POS
2. merge POS + logistik (left join)
3. merge + CRM (left join)

Alasan left join: transaksi tanpa pengiriman dan pelanggan non aktif tetap harus dihitung.

In [9]:
print('duplikat logistik:', df_logistik.duplicated(subset='order_id').sum())
print('duplikat POS:', df_pos.duplicated(subset=['InvoiceNo', 'StockCode']).sum())

duplikat logistik: 0
duplikat POS: 1965


In [10]:
df_logistik = df_logistik.drop_duplicates(subset='order_id')
df_pos = df_pos.drop_duplicates(subset=['InvoiceNo', 'StockCode'])

df_master_analisis = df_pos.merge(
    df_logistik,
    how='left',
    left_on='InvoiceNo',
    right_on='order_id'
).merge(df_crm, how='left', on='customer_id')

df_master_analisis = df_master_analisis.drop(columns=['order_id'])
print(df_master_analisis.shape)
df_master_analisis.head()

(78035, 25)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total_amount,coupon_code,...,shipment_service,timeline_shipped_at,timeline_delivered_at,email,gender,age,city,membership_tier,loyalty_points,is_active
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,NaN,...,YES,2011-02-13 00:00:00,2011-02-16 00:00:00,customer17850@mail.com,L,37.0,Jakarta,Gold,2313.0,1.0


Jawaban untuk Data Owner

In [11]:
kupon = df_master_analisis[df_master_analisis['coupon_code'].notna()]
inv = kupon.drop_duplicates(subset='InvoiceNo')

print('Total transaksi pakai kupon :', len(inv), 'invoice')
print('Total nilai transaksi       :', round(kupon['total_amount'].sum(), 2))
print()
print('Profil loyalitas pemakai kupon:')
print(inv['membership_tier'].value_counts())
print()
print('Status pengiriman:')
print(inv['status'].value_counts())
print()
inv_kirim = inv.dropna(subset=['status'])
terkirim = (inv_kirim['status'] == 'Delivered').sum()
print(f'Terkirim: {terkirim} dari {len(inv_kirim)} ({terkirim/len(inv_kirim)*100:.1f}%)')

Total transaksi pakai kupon : 203 invoice
Total nilai transaksi       : 68949.57

Profil loyalitas pemakai kupon:
membership_tier
Silver      51
Bronze      49
Gold        46
Platinum    43
Name: count, dtype: int64

Status pengiriman:
status
Delivered     153
In Transit     25
Shipped        21
Cancelled       4
Name: count, dtype: int64

Terkirim: 153 dari 203 (75.4%)


Bagian C.1

In [12]:
def generate_data_dictionary(df):
    kamus = pd.DataFrame({
        'nama_kolom': df.columns,
        'tipe_data_sistem': [str(t) for t in df.dtypes],
        'jumlah_missing': df.isna().sum().values,
        'persen_missing': (df.isna().mean().values * 100).round(2),
        'jumlah_nilai_unik': df.nunique(dropna=True).values,
    })

    sampel = []
    for col in df.columns:
        nilai = df[col].dropna().unique()[:3]
        sampel.append(', '.join(str(v) for v in nilai))
    kamus['sampel_nilai'] = sampel

    return kamus


kamus = generate_data_dictionary(df_master_analisis)
kamus.to_csv('kamus_data_hasil.csv', index=False)
kamus

,nama_kolom,tipe_data_sistem,jumlah_missing,persen_missing,jumlah_nilai_unik,sampel_nilai
0,InvoiceNo,str,0,0.00,4005,"536365, 536366, 536367"
1,StockCode,str,0,0.00,2832,"85123A, 71053, 84406B"
2,Description,str,0,0.00,2860,"WHITE HANGING HEART T-LIGHT HOLDER, WHITE META..."
3,Quantity,int64,0,0.00,154,"6, 8, 2"
4,InvoiceDate,datetime64[us],0,0.00,3796,"2010-12-01 08:26:00, 2010-12-01 08:28:00, 2010..."
5,UnitPrice,float64,0,0.00,188,"2.55, 3.39, 2.75"
6,CustomerID,int64,0,0.00,1904,"17850, 13047, 12583"
7,Country,str,0,0.00,30,"United Kingdom, France, Australia"
8,total_amount,float64,0,0.00,1229,"15.3, 20.34, 22.0"
9,coupon_code,str,73988,94.81,4,"NRMFLASH15, AKHIRBULAN25, DISKON20"


Bagian C.2

In [13]:
duplikat = df_master_analisis.duplicated(subset=['InvoiceNo', 'StockCode']).sum()
assert duplikat == 0, f'ada {duplikat} duplikat'
print('cek 1 lolos - tidak ada duplikat primary key')

negatif = (df_master_analisis['total_amount'] < 0).sum()
assert negatif == 0, f'ada {negatif} nilai negatif'
print('cek 2 lolos - tidak ada total_amount negatif')

future = (df_master_analisis['InvoiceDate'] > pd.Timestamp.now()).sum()
assert future == 0, f'ada {future} future date'
print('cek 3 lolos - tidak ada future date')

print()
print('semua validasi lolos')

cek 1 lolos - tidak ada duplikat primary key
cek 2 lolos - tidak ada total_amount negatif
cek 3 lolos - tidak ada future date

semua validasi lolos
